# 03 — Race Prediction from Last Names

Predicts race category from a surname using a **character-frequency feature matrix**.

**Target Classes:** white | hispanic | asian | black | americanindian

**Models Compared:**
1. Logistic Regression
2. K-Nearest Neighbors
3. Naive Bayes (Multinomial)
4. Random Forest ← *best, saved to models/*
5. Support Vector Machine

**Data:** `data/processed/Modified_Race_last_name.csv` — 162,251 Census surnames

> **Note on class imbalance:** ~82% of surnames are labelled *white*. Precision/recall for minority classes (americanindian, black, asian) will be lower. This is an inherent limitation of the data distribution.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE       = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROCESSED  = os.path.join(BASE, 'data', 'processed')
MODELS_DIR = os.path.join(BASE, 'models')

RANDOM_SEED = 42
print('Base:', BASE)

## 1. Load Data

In [ ]:
df = pd.read_csv(os.path.join(PROCESSED, 'Modified_Race_last_name.csv'))
print(f'Shape: {df.shape}')
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
# Identify name and race columns
name_col = df.columns[0]
race_col = df.columns[1]
print(f'Name column: "{name_col}"  |  Race column: "{race_col}"')

print('\nClass distribution:')
race_counts = df[race_col].value_counts()
print(race_counts)
print(f'\nClass imbalance ratio: {race_counts.max()/race_counts.min():.1f}x')

In [ ]:
# Drop rows with missing values
df = df.dropna(subset=[name_col, race_col]).reset_index(drop=True)
print(f'After dropping nulls: {len(df):,} rows')

## 2. Feature Engineering — Letter Frequency Matrix

In [ ]:
LETTERS = list('abcdefghijklmnopqrstuvwxyz')

def name_to_features(name: str) -> list:
    """Return a 26-dim vector of letter frequencies for a given surname."""
    name = str(name).lower()
    return [name.count(ch) for ch in LETTERS]

X_raw = np.array([name_to_features(n) for n in df[name_col]])

# Encode labels
le = LabelEncoder()
y  = le.fit_transform(df[race_col])

print(f'Feature matrix shape : {X_raw.shape}')
print(f'Classes              : {le.classes_}')

In [ ]:
# Heat-map of mean letter frequency per race
feat_df = pd.DataFrame(X_raw, columns=LETTERS)
feat_df['Race'] = df[race_col].values
heat = feat_df.groupby('Race')[LETTERS].mean()

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(heat, cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Mean Letter Frequency per Surname — by Race')
ax.set_xlabel('Letter')
plt.tight_layout()
plt.show()

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

## 4. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes':         MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=20,
                                                   random_state=RANDOM_SEED, n_jobs=-1),
    'SVM (RBF)':           SVC(kernel='rbf', C=10, random_state=RANDOM_SEED),
}

results = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred   = clf.predict(X_test)
    acc      = accuracy_score(y_test, y_pred)
    results[name] = {'model': clf, 'accuracy': acc, 'y_pred': y_pred}
    print(f'{name:<25}  accuracy = {acc:.4f}')

## 5. Cross-Validation on Best Model (Random Forest)

In [ ]:
best_clf = results['Random Forest']['model']

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(best_clf, X_raw, y, cv=cv, scoring='accuracy', n_jobs=-1)

print(f'10-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Fold scores: {cv_scores.round(4)}')

## 6. Model Comparison Plot

In [ ]:
acc_values = {k: v['accuracy'] for k, v in results.items()}
sorted_acc = dict(sorted(acc_values.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if k == 'Random Forest' else '#3498db' for k in sorted_acc]
bars = ax.barh(list(sorted_acc.keys()), list(sorted_acc.values()), color=colors)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('Test Accuracy')
ax.set_title('Race Prediction — Model Comparison')
for bar, val in zip(bars, sorted_acc.values()):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'docs', 'race_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Detailed Evaluation — Random Forest

In [ ]:
y_pred_rf = results['Random Forest']['y_pred']
print('Classification Report:')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=30)
ax.set_title('Confusion Matrix — Race (Random Forest)')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'docs', 'race_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class accuracy to highlight imbalance impact
per_class_acc = cm.diagonal() / cm.sum(axis=1)
print('Per-class accuracy:')
for cls, acc in zip(le.classes_, per_class_acc):
    print(f'  {cls:<16} {acc:.4f}')

## 8. Save Best Model

In [ ]:
model_path = os.path.join(MODELS_DIR, 'race_prediction_model_rf.pkl')
le_path    = os.path.join(MODELS_DIR, 'race_label_encoder.pkl')

if os.path.exists(model_path):
    print(f'Model already saved at: {model_path}')
    print('To retrain and overwrite, delete the file and re-run.')
else:
    joblib.dump(best_clf, model_path)
    print(f'Model saved: {model_path}')

# Always save the label encoder alongside the model
joblib.dump(le, le_path)
print(f'Label encoder saved: {le_path}')

## 9. Testing — Load Model & Predict

In [ ]:
# Load saved model and label encoder
loaded_model = joblib.load(model_path)
loaded_le    = joblib.load(le_path)
print('Model and label encoder loaded.')
print('Classes:', loaded_le.classes_)

def predict_race(last_name: str, model, label_enc) -> str:
    """Return predicted race category for a given surname."""
    features = np.array([name_to_features(last_name)])
    pred_idx = model.predict(features)[0]
    return label_enc.inverse_transform([pred_idx])[0]

# Test cases (approximate expected labels based on Census surname data)
test_names = [
    ('Smith',       'white'),
    ('Garcia',      'hispanic'),
    ('Kim',         'asian'),
    ('Washington',  'black'),
    ('Johnson',     'white'),
    ('Rodriguez',   'hispanic'),
    ('Chen',        'asian'),
    ('Williams',    'black'),
]

print(f'\n{"Surname":<16} {"Expected":<14} {"Predicted":<14} {"Correct"}')
print('-' * 55)
correct = 0
for name, expected in test_names:
    pred = predict_race(name, loaded_model, loaded_le)
    ok   = pred == expected
    correct += ok
    print(f'{name:<16} {expected:<14} {pred:<14} {"✓" if ok else "✗"}')

print(f'\nTest accuracy: {correct}/{len(test_names)} ({correct/len(test_names)*100:.1f}%)')

In [ ]:
# Probability output for each test name
print(f'{"Surname":<16}', '  '.join(f'{c[:6]:>8}' for c in loaded_le.classes_))
print('-' * 80)
for name, _ in test_names:
    feats = np.array([name_to_features(name)])
    probs = loaded_model.predict_proba(feats)[0]
    prob_str = '  '.join(f'{p:>8.3f}' for p in probs)
    print(f'{name:<16} {prob_str}')